# Rangkuman Kinerja Proyek NIDS Lintas-Jaringan (SFM + XGBoost)

**Untuk presentasi / PPT.** Notebook ini merangkum seluruh hasil eksperimen nyata proyek:
dari dua dataset sumber, pembersihan data, pemetaan fitur (SFM), pelatihan & pengujian model,
adaptasi domain, evaluasi adversarial, efisiensi *edge*, hingga validasi trafik nyata (FAR) di AWS.

> **Prinsip kejujuran data:** semua angka berasal dari eksperimen nyata (berkas `*.json` di folder induk
> dan hasil capture AWS). Bila berkas JSON tersedia, notebook memuatnya; bila tidak, dipakai nilai
> *fallback* yang identik dengan hasil tercatat sehingga notebook tetap jalan di mana pun (mis. SageMaker).

Jalankan sel berurutan dari atas ke bawah.

## 0. Setup & pemuatan hasil

Jalankan sel instalasi di bawah **sekali** bila kernel belum punya paket (mis. error
`No module named 'matplotlib'`). Setelah instalasi selesai, lanjutkan ke sel berikutnya
(tak perlu restart untuk `%pip install`).

In [ ]:
# Instalasi paket bila belum ada (aman dijalankan berulang).
import importlib, sys, subprocess
need = [m for m in ('matplotlib', 'pandas', 'numpy') if importlib.util.find_spec(m) is None]
if need:
    print('Menginstal:', need)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
    print('Selesai. Bila import di sel berikutnya masih gagal, Restart Kernel lalu jalankan lagi.')
else:
    print('Semua paket sudah tersedia.')

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

# Cari folder berkas hasil JSON (folder induk dari notebooks/, atau folder saat ini)
CANDIDATES = ['..', '.', '../unswnb-15', 'unswnb-15']
DATA_DIR = next((d for d in CANDIDATES if os.path.exists(os.path.join(d, 'cross_dataset_baseline.json'))), '..')
print('DATA_DIR =', os.path.abspath(DATA_DIR))

def load_json(name, fallback=None):
    p = os.path.join(DATA_DIR, name)
    if os.path.exists(p):
        with open(p) as f:
            print('  loaded:', name)
            return json.load(f)
    print('  (fallback):', name)
    return fallback

## 1. Dua Dataset Sumber

Proyek ini sengaja memakai **pasangan dataset dari sumber & alat ekstraksi berbeda** untuk menguji
generalisasi lintas-jaringan secara jujur:

- **CSE-CIC-IDS2018** — trafik *testbed* CIC (2018), diekstraksi dengan **CICFlowMeter**.
- **UNSW-NB15** — trafik dibangkitkan IXIA PerfectStorm (2015), diekstraksi dengan **Argus + Bro/Zeek**.

Perbedaan sumber & ekstraktor ini bukan kelemahan, melainkan prasyarat menguji *cross-network robustness*.

In [ ]:
inv = load_json('feature_inventory.json', fallback={
    'cic_ids2018': {'n_features': 68, 'extractor': 'CICFlowMeter'},
    'unsw_nb15':   {'n_features': 42, 'extractor': 'Argus + Bro/Zeek (+12 custom algorithms)'}
})

# Tabel perbandingan dua dataset (angka nyata dari proyek)
compare = pd.DataFrame([
    {'Aspek': 'Tahun / sumber',        'CSE-CIC-IDS2018': '2018, testbed CIC',       'UNSW-NB15': '2015, IXIA PerfectStorm'},
    {'Aspek': 'Alat ekstraksi',        'CSE-CIC-IDS2018': inv['cic_ids2018']['extractor'], 'UNSW-NB15': inv['unsw_nb15']['extractor']},
    {'Aspek': 'Jumlah fitur',          'CSE-CIC-IDS2018': inv['cic_ids2018']['n_features'], 'UNSW-NB15': inv['unsw_nb15']['n_features']},
    {'Aspek': 'Skema label',           'CSE-CIC-IDS2018': 'Benign + 14 jenis serangan', 'UNSW-NB15': 'Normal + 9 jenis serangan'},
    {'Aspek': 'Record dipakai (biner)','CSE-CIC-IDS2018': '1.348.453 normal / 274.808 attack', 'UNSW-NB15': '175.341 latih / 82.332 uji'},
    {'Aspek': 'Granularitas',          'CSE-CIC-IDS2018': 'per-flow', 'UNSW-NB15': 'per-flow'},
])
compare

In [ ]:
# Jenis serangan di kedua dataset (untuk konteks slide)
cic_attacks = ['Benign', 'DoS (Hulk/GoldenEye/Slowloris/SlowHTTPTest)', 'DDoS (LOIC/HOIC)',
               'Brute-Force (FTP/SSH)', 'Web (XSS/SQLi/Brute)', 'Infiltration', 'Botnet (Ares)']
unsw_attacks = ['Normal', 'Generic', 'Exploits', 'Fuzzers', 'DoS', 'Reconnaissance',
                'Analysis', 'Backdoor', 'Shellcode', 'Worms']
print('CSE-CIC-IDS2018 (kelompok serangan):')
for a in cic_attacks: print('   -', a)
print('\nUNSW-NB15 (10 kelas):')
for a in unsw_attacks: print('   -', a)
print('\nUntuk tahap ini keduanya dibinerkan: attack vs normal.')

In [ ]:
# Diagram batang: jumlah fitur per dataset
fig, ax = plt.subplots(figsize=(5.5, 3.2))
names = ['CSE-CIC-IDS2018\n(CICFlowMeter)', 'UNSW-NB15\n(Argus+Bro/Zeek)']
vals = [inv['cic_ids2018']['n_features'], inv['unsw_nb15']['n_features']]
bars = ax.bar(names, vals, color=['#4C72B0', '#DD8452'])
ax.set_ylabel('Jumlah fitur')
ax.set_title('Perbedaan jumlah fitur antar-ekstraktor')
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+1, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Pembersihan Data & Binerisasi

Langkah pra-pemrosesan utama:
1. **Buang baris rusak** (inf / NaN dari pembagian, mis. `Flow Byts/s` saat durasi 0).
2. **Binerisasi label**: semua jenis serangan -> `attack` (1), sisanya `normal` (0).
3. **Verifikasi label** pada CIC: `Benign`=0 menghasilkan tepat 1.348.453 normal & 274.808 attack.
4. **StandardScaler (z-score) per-dataset terpisah** agar perbedaan satuan/skala ternormalisasi
   tanpa mengarang konversi antar-alat.

In [ ]:
# Komposisi kelas biner kedua dataset (angka verifikasi nyata)
# CIC: total populasi biner terverifikasi. UNSW: diturunkan dari confusion same_unsw (test set).
cic_normal, cic_attack = 1348453, 274808
try:
    cm = base['results']['model_A']['same_unsw']['confusion']  # [[TN,FP],[FN,TP]] pd test UNSW
    unsw_normal = cm[0][0] + cm[0][1]
    unsw_attack = cm[1][0] + cm[1][1]
except Exception:
    unsw_normal, unsw_attack = 37000, 45332  # fallback (test set UNSW)

fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.4))
for ax, (title, nrm, atk) in zip(axes, [
        (f'CSE-CIC-IDS2018\n(total {cic_normal+cic_attack:,} flow)', cic_normal, cic_attack),
        (f'UNSW-NB15 (test set)\n(total {unsw_normal+unsw_attack:,} flow)', unsw_normal, unsw_attack)]):
    ax.pie([nrm, atk], labels=['normal', 'attack'], autopct='%1.1f%%',
           colors=['#55A868', '#C44E52'], startangle=90, explode=(0, 0.05))
    ax.set_title(title)
plt.suptitle('Komposisi kelas pasca-binerisasi', fontsize=12)
plt.tight_layout(); plt.show()
print('Rasio ketidakseimbangan (normal:attack):')
print('  CIC  = %.1f : 1  (mayoritas normal)' % (cic_normal/cic_attack))
print('  UNSW = 1 : %.1f  (mayoritas attack -> distribusi berbeda dari CIC)' % (unsw_attack/unsw_normal))

## 3. Semantic Feature Mapping (SFM) & Validasi

SFM memetakan fitur berfungsi-sama antar-dataset meski nama kolomnya berbeda
(mis. `Flow Duration` <-> `dur`). Tiap pasangan divalidasi statistik (rentang, distribusi, satuan),
**bukan** sekadar kemiripan nama. Verdict:
- `aligned` — langsung sepadan.
- `scale-mismatch` — sepadan tapi perlu penskalaan (ditangani z-score per-dataset).
- `likely-different-feature` — **dibuang** (mis. TCP window & IAT), studi kasus *feature-extractor mismatch*.

In [ ]:
mapv = load_json('mapping_validation.json', fallback=[])
if mapv:
    dfm = pd.DataFrame(mapv)[['cic', 'unsw', 'hyp', 'verdict']]
    dfm.columns = ['Fitur CIC', 'Fitur UNSW', 'Hipotesis', 'Verdict']
    display(dfm)
    print('\nRingkasan verdict:')
    print(pd.Series([m['verdict'] for m in mapv]).value_counts().to_string())
else:
    print('mapping_validation.json tidak ditemukan.')

**Himpunan fitur final (Model A, 9 fitur irisan kuat):**
`duration, fwd_pkts, bwd_pkts, fwd_bytes, bwd_bytes, fwd_mean, bwd_mean, src_load, dst_load`.
Model B menambah `fwd_iat, bwd_iat` (11 fitur) — setara Model A namun kurang ringkas.

## 4. Pelatihan & Pengujian: Celah Generalisasi Lintas-Jaringan

Model: **XGBoost** (`max_depth=8, lr=0.1, n_estimators=200, subsample/colsample=0.8, binary:logistic`).
Metrik utama **MCC** (tahan *class imbalance*). Empat skenario diuji.

In [ ]:
base = load_json('cross_dataset_baseline.json', fallback={'results': {'model_A': {
    'same_cic': {'mcc': 0.9134, 'f1': 0.9277}, 'same_unsw': {'mcc': 0.7448, 'f1': 0.8904},
    'cic2unsw': {'mcc': -0.0719, 'f1': 0.0224}, 'unsw2cic': {'mcc': -0.0613, 'f1': 0.0088}}}})
rA = base['results']['model_A']
scen = ['Same-CIC', 'Same-UNSW', 'CIC->UNSW', 'UNSW->CIC']
keys = ['same_cic', 'same_unsw', 'cic2unsw', 'unsw2cic']
mccs = [rA[k]['mcc'] for k in keys]
f1s  = [rA[k].get('f1', float('nan')) for k in keys]

# Tabel ringkas MCC + F1
tbl = pd.DataFrame({'Skenario': scen, 'MCC': np.round(mccs, 4), 'F1': np.round(f1s, 4)})
display(tbl)

# Grouped bar: MCC vs F1 per skenario
x = np.arange(len(scen)); ww = 0.38
fig, ax = plt.subplots(figsize=(7.2, 3.8))
b1 = ax.bar(x-ww/2, mccs, ww, label='MCC', color='#4C72B0')
b2 = ax.bar(x+ww/2, f1s,  ww, label='F1',  color='#DD8452')
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(scen)
ax.set_ylabel('Skor'); ax.set_ylim(-0.2, 1.0)
ax.set_title('Model A: in-domain tinggi, lintas-jaringan runtuh (~0)')
for bars in (b1, b2):
    for b in bars:
        v = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, v + (0.02 if v>=0 else -0.07), f'{v:.2f}', ha='center', fontsize=8)
ax.legend(); plt.tight_layout(); plt.show()
print('Baik MCC maupun F1 runtuh lintas-jaringan; generalization gap MCC ~0.81-0.99.')
print('Catatan: F1 lintas-jaringan sangat rendah (mendekati 0) -> model gagal mengenali kelas attack di jaringan asing.')

## 5. Diagnosis & Solusi: Distribution Shift, Bukan Kekurangan Fitur

- **Joint training** (gabung CIC+UNSW) mencapai MCC hampir setara in-domain di kedua jaringan
  serentak -> membuktikan **SFM valid** (9 fitur cukup ekspresif). Maka celah = *distribution shift*.
- **Few-shot 1% label target** memulihkan MCC dari negatif ke 0.65-0.90.
- **Mixup** (tanpa label target) juga memulihkan sebagian besar.

In [ ]:
da = load_json('domain_adaptation.json', fallback=None)
align = load_json('cross_network_alignment.json', fallback=None)

if da:
    fs_c = pd.DataFrame(da['fewshot']['cic2unsw'])
    fs_u = pd.DataFrame(da['fewshot']['unsw2cic'])

    def fmt(df):
        d = df.copy()
        d['frac'] = (d['frac']*100).map(lambda v: f'{v:g}%')
        cols = [c for c in ['frac','n_target','mcc','f1','acc'] if c in d.columns]
        d = d[cols].round(4)
        d.columns = ['Fraksi label target','n_flow target','MCC','F1','Akurasi'][:len(cols)]
        return d

    # DUA TABEL TERPISAH karena kedua arah berbeda hasil (asimetris)
    print('=== Tabel 1: CIC -> UNSW (latih CIC + x% UNSW, uji UNSW) ===')
    display(fmt(fs_c))
    print('\n=== Tabel 2: UNSW -> CIC (latih UNSW + x% CIC, uji CIC) ===')
    display(fmt(fs_u))

    # Kurva perbandingan kedua arah
    fig, ax = plt.subplots(figsize=(6.6, 3.8))
    ax.plot(fs_c['frac']*100, fs_c['mcc'], 'o-', label='CIC->UNSW', color='#4C72B0')
    ax.plot(fs_u['frac']*100, fs_u['mcc'], 's-', label='UNSW->CIC', color='#DD8452')
    ax.axhline(0, color='k', lw=0.8, ls=':')
    ax.set_xlabel('Fraksi label target (%)'); ax.set_ylabel('MCC lintas-jaringan')
    ax.set_title('Few-shot: 1% label target sudah memulihkan MCC (asimetris antar-arah)')
    ax.legend(); plt.tight_layout(); plt.show()

    print('Asimetri: UNSW->CIC pulih jauh lebih tinggi (0%%=%.3f -> 1%%=%.3f, ~in-domain)'
          % (fs_u.iloc[0]['mcc'], fs_u.iloc[1]['mcc']))
    print('          CIC->UNSW pulih lebih rendah      (0%%=%.3f -> 1%%=%.3f)'
          % (fs_c.iloc[0]['mcc'], fs_c.iloc[1]['mcc']))
    print('Mixup (tanpa label target): CIC->UNSW MCC=%.3f ; UNSW->CIC MCC=%.3f' %
          (da['mixup']['cic2unsw']['mcc'], da['mixup']['unsw2cic']['mcc']))
else:
    print('domain_adaptation.json tidak ditemukan.')

In [ ]:
# Tabel strategi penyelarasan (baseline / CORAL / few-shot / mixup / joint).
# Kolom disamakan berdasarkan 'diuji di jaringan mana', BUKAN arah transfer,
# agar baris joint (dilatih di keduanya) tidak menyesatkan.
align = align if 'align' in dir() else load_json('cross_network_alignment.json', fallback=None)
da = da if 'da' in dir() else load_json('domain_adaptation.json', fallback=None)
if align:
    cs = {r['strategi']: r for r in align['cross_summary']}
    def get(strat, direction):
        return cs.get(strat, {}).get(direction, float('nan'))
    rows = [
        {'Strategi': 'Baseline single-source (0% target)',
         'MCC di test UNSW': get('Baseline single-source', 'cic2unsw'),
         'MCC di test CIC':  get('Baseline single-source', 'unsw2cic')},
        {'Strategi': 'CORAL alignment',
         'MCC di test UNSW': get('CORAL alignment', 'cic2unsw'),
         'MCC di test CIC':  get('CORAL alignment', 'unsw2cic')},
    ]
    # Sisipkan baris few-shot (1/5/10/25%) dari domain_adaptation.json bila tersedia.
    # MCC di test UNSW <- arah cic2unsw ; MCC di test CIC <- arah unsw2cic.
    if da:
        fc = {round(r['frac'], 4): r['mcc'] for r in da['fewshot']['cic2unsw']}
        fu = {round(r['frac'], 4): r['mcc'] for r in da['fewshot']['unsw2cic']}
        for frac in [0.01, 0.05, 0.10, 0.25]:
            rows.append({'Strategi': f'Few-shot {int(frac*100)}% label target',
                         'MCC di test UNSW': fc.get(frac, float('nan')),
                         'MCC di test CIC':  fu.get(frac, float('nan'))})
        rows.append({'Strategi': 'Mixup (tanpa label target)',
                     'MCC di test UNSW': da['mixup']['cic2unsw']['mcc'],
                     'MCC di test CIC':  da['mixup']['unsw2cic']['mcc']})
    rows.append({'Strategi': 'Joint training (latih di keduanya, 100%)',
                 'MCC di test UNSW': align['joint']['unsw_test_mcc'],
                 'MCC di test CIC':  align['joint']['cic_test_mcc']})
    tbl = pd.DataFrame(rows).round(4)
    display(tbl)
    print('Kolom = performa pada test set jaringan tsb (bukan arah transfer).')
    print('Alur pemulihan: baseline (~0) -> few-shot 1% (lompat) -> mendatar -> joint (batas atas).')
else:
    print('cross_network_alignment.json tidak ditemukan.')

### 5b. Verifikasi kuantitatif: Jarak Wasserstein turun setelah kalibrasi

In [ ]:
w = load_json('wasserstein_shift.json', fallback=None)
if w:
    def row(direction):
        r = w['results'][direction]
        return {'Arah': direction,
                'Baseline': r['before']['mean'],
                'Few-shot 1%': r['fewshot_1pct']['mean'],
                'Mixup': r['mixup']['mean'],
                'Batas-bawah': r['target_train_lb']['mean']}
    dw = pd.DataFrame([row('cic2unsw'), row('unsw2cic')])
    display(dw)
    print('W1 (rata-rata 9 fitur) mengecil ke arah target -> kalibrasi benar menggeser distribusi.')
else:
    print('wasserstein_shift.json tidak ditemukan.')

## 6. Evaluasi Adversarial yang Realistis

> **Istilah (konsisten dgn paper):** *adversarial training* = fase MELATIH model dgn sampel
> adversarial (menghasilkan model **robust**). *Evasion* = fase MENGUJI/menyerang model saat
> inferensi (data uji dimodifikasi jadi adversarial). Di bawah, serangan uji disebut **evasion**.

- **Functional-preserving evasion**: serangan dibatasi agar tetap valid protokol -> ancaman lebih
  realistis (tak sebesar FGSM tak-terbatas yang menghasilkan flow mustahil).
- **Evasion transfer (grey-box)**: sampel evasion dibuat pakai model pengganti -> model robust
  tampak sangat tahan (MCC ~0.99).
- **Evasion adaptive (white-box)**: sampel evasion dibuat pakai model target sendiri -> ketahanan
  **runtuh** (MCC 0.25-0.43). Ini koreksi jujur atas klaim ketahanan berlebih.
- **Baseline vs Robust**: model baseline (tanpa adversarial training) runtuh saat kena evasion;
  robust bertahan sebagian -> menegaskan manfaat sekaligus batas adversarial training.

In [ ]:
awb = load_json('adaptive_whitebox.json', fallback=None)
if awb:
    s = {d['arah']: d for d in awb['summary_eps01']}
    R = awb.get('results', {})

    # ---- TABEL 1: MODEL ROBUST (adversarial-trained) @eps=0.1 ----
    tbl = pd.DataFrame([
        {'Jaringan': 'CIC',
         'clean': s['cic']['robust_clean'],
         'evasion transfer (grey-box)': s['cic']['robust_transfer'],
         'evasion adaptive (white-box)': s['cic']['robust_adaptive'],
         'drop adaptive': s['cic']['drop_adaptive']},
        {'Jaringan': 'UNSW',
         'clean': s['unsw']['robust_clean'],
         'evasion transfer (grey-box)': s['unsw']['robust_transfer'],
         'evasion adaptive (white-box)': s['unsw']['robust_adaptive'],
         'drop adaptive': s['unsw']['drop_adaptive']},
    ])
    tbl_disp = tbl.copy()
    for c in ['clean','evasion transfer (grey-box)','evasion adaptive (white-box)']:
        tbl_disp[c] = tbl_disp[c].round(4)
    tbl_disp['drop adaptive'] = (tbl_disp['drop adaptive']*100).round(1).astype(str) + '%'
    print('Tabel 1 - MODEL ROBUST (adversarial-trained), MCC @eps=0.1:')
    display(tbl_disp)

    # ---- TABEL 2: BASELINE vs ROBUST (clean vs evasion adaptive) ----
    def mcc(net, key):
        return R.get(net, {}).get(key, {}).get('mcc', float('nan'))
    base_tbl = pd.DataFrame([
        {'Jaringan': 'CIC', 'Model': 'Baseline (tanpa adv-training)',
         'clean': mcc('cic','baseline_clean'), 'evasion adaptive': mcc('cic','baseline_adaptive_eps0.1')},
        {'Jaringan': 'CIC', 'Model': 'Robust (adv-trained)',
         'clean': mcc('cic','robust_clean'), 'evasion adaptive': mcc('cic','robust_adaptive_eps0.1')},
        {'Jaringan': 'UNSW', 'Model': 'Baseline (tanpa adv-training)',
         'clean': mcc('unsw','baseline_clean'), 'evasion adaptive': mcc('unsw','baseline_adaptive_eps0.1')},
        {'Jaringan': 'UNSW', 'Model': 'Robust (adv-trained)',
         'clean': mcc('unsw','robust_clean'), 'evasion adaptive': mcc('unsw','robust_adaptive_eps0.1')},
    ]).round(4)
    print('\nTabel 2 - BASELINE vs ROBUST (MCC, clean vs evasion adaptive @eps=0.1):')
    display(base_tbl)

    # ---- GRAFIK A: robust vs 3 situasi ----
    labels = ['clean', 'evasion\ntransfer\n(grey-box)', 'evasion\nadaptive\n(white-box)']
    cic = [s['cic']['robust_clean'], s['cic']['robust_transfer'], s['cic']['robust_adaptive']]
    unsw = [s['unsw']['robust_clean'], s['unsw']['robust_transfer'], s['unsw']['robust_adaptive']]
    x = np.arange(len(labels)); ww = 0.36
    fig, ax = plt.subplots(figsize=(6.6, 3.9))
    b1 = ax.bar(x-ww/2, cic, ww, label='CIC', color='#4C72B0')
    b2 = ax.bar(x+ww/2, unsw, ww, label='UNSW', color='#DD8452')
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel('MCC (robust model, eps=0.1)'); ax.set_ylim(0, 1.05)
    ax.set_title('Model robust: tampak kuat vs transfer, runtuh vs adaptive')
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{b.get_height():.2f}', ha='center', fontsize=8)
    ax.legend(); plt.tight_layout(); plt.show()

    # ---- GRAFIK B: baseline vs robust (clean vs evasion adaptive) ----
    grp = ['CIC clean','CIC evasion','UNSW clean','UNSW evasion']
    base_v = [mcc('cic','baseline_clean'), mcc('cic','baseline_adaptive_eps0.1'),
              mcc('unsw','baseline_clean'), mcc('unsw','baseline_adaptive_eps0.1')]
    rob_v  = [mcc('cic','robust_clean'), mcc('cic','robust_adaptive_eps0.1'),
              mcc('unsw','robust_clean'), mcc('unsw','robust_adaptive_eps0.1')]
    x = np.arange(len(grp)); ww = 0.38
    fig, ax = plt.subplots(figsize=(7.6, 3.9))
    ax.bar(x-ww/2, base_v, ww, label='Baseline', color='#C44E52')
    ax.bar(x+ww/2, rob_v,  ww, label='Robust',   color='#55A868')
    ax.set_xticks(x); ax.set_xticklabels(grp)
    ax.set_ylabel('MCC (eps=0.1)'); ax.set_ylim(0, 1.05)
    ax.set_title('Baseline vs Robust: clean vs evasion adaptive')
    ax.legend(); plt.tight_layout(); plt.show()

    print('Ringkas: baseline runtuh saat evasion; robust tahan vs transfer TAPI tetap jebol vs adaptive.')
    print('Drop adaptive (robust): CIC -%.1f%% ; UNSW -%.1f%% (rasa aman keliru bila hanya uji transfer).'
          % (s['cic']['drop_adaptive']*100, s['unsw']['drop_adaptive']*100))
else:
    print('adaptive_whitebox.json tidak ditemukan.')

## 7. Efisiensi Model (Edge / Green AI)

Diukur nyata pada 1 vCPU (*single-thread*, batch=1) untuk meniru penyebaran *edge*.

In [ ]:
eff = load_json('model_efficiency.json', fallback={
    'size_kb': 2961.8, 'latency_per_flow_us': {'mean': 439.3, 'median': 434.2, 'std': 22.9},
    'throughput_flows_per_sec': {'incremental': 2276}})
eff_tbl = pd.DataFrame([
    {'Metrik': 'Ukuran model biner (9 fitur, 200 pohon)', 'Nilai': f"{eff['size_kb']/1024:.1f} MB"},
    {'Metrik': 'Latensi inferensi per flow (median, 1 vCPU)', 'Nilai': f"{eff['latency_per_flow_us']['median']:.0f} us"},
    {'Metrik': 'Latensi inferensi per flow (mean +/- std)', 'Nilai': f"{eff['latency_per_flow_us']['mean']:.0f} +/- {eff['latency_per_flow_us']['std']:.0f} us"},
    {'Metrik': 'Throughput inkremental (1 vCPU)', 'Nilai': f"~{eff['throughput_flows_per_sec']['incremental']:,} flow/detik"},
])
eff_tbl

## 8. Validasi Trafik Nyata di AWS — False Alarm Rate (FAR)

Pipeline: **tcpdump (pcap) -> NFStream (9 fitur SFM) -> XGBoost**. Fase 1 = trafik *benign* saja,
sehingga tiap prediksi `attack` = alarm palsu. `FAR = false_alarm / total_flow`.

Ramp durasi bertahap membuktikan FAR rendah **stabil**, bukan artefak cuplikan pendek.
Notebook mencoba memuat `far_log.jsonl` (bila dijalankan di mesin AWS / diunduh dari S3);
bila tak ada, dipakai hasil tercatat sesi terakhir.

In [ ]:
# Coba muat far_log.jsonl (real-traffic). Cari di beberapa lokasi umum.
far_paths = ['/opt/unsw/results/far_log.jsonl', os.path.join(DATA_DIR, 'far_log.jsonl'), 'far_log.jsonl']
far_rows = None
for p in far_paths:
    if os.path.exists(p):
        far_rows = [json.loads(l) for l in open(p) if l.strip()]
        print('loaded far_log.jsonl dari', p); break

if not far_rows:
    print('(fallback) memakai hasil tercatat S0 & D1.')
    far_rows = [
        {'pcap': 'ramp_s0 (3 menit)', 'n_flow': 203,  'n_false_alarm': 0,  'far': 0.0},
        {'pcap': 'D1 (1 jam)',        'n_flow': 3687, 'n_false_alarm': 14, 'far': 0.003797},
    ]

dff = pd.DataFrame(far_rows)
cols = [c for c in ['pcap','n_flow','n_false_alarm','far'] if c in dff.columns]
dff = dff[cols].copy()
dff['FAR (%)'] = (dff['far']*100).round(4)
display(dff)

In [ ]:
# Tabel perbandingan FAR antar-durasi (isi otomatis dari D1..D5 bila tersedia)
ramp = pd.DataFrame([
    {'Tahap': 'S0 (3 menit)', 'n_flow': 203,  'false_alarm': 0,  'FAR (%)': 0.0000, 'Status': 'gate LOLOS'},
    {'Tahap': 'D1 (1 jam)',   'n_flow': 3687, 'false_alarm': 14, 'FAR (%)': 0.3797, 'Status': 'gate LOLOS'},
    {'Tahap': 'D2 (6 jam)',   'n_flow': None, 'false_alarm': None, 'FAR (%)': None,  'Status': 'berjalan'},
    {'Tahap': 'D3 (24 jam)',  'n_flow': None, 'false_alarm': None, 'FAR (%)': None,  'Status': 'menyusul'},
])
display(ramp)

sub = ramp.dropna(subset=['FAR (%)'])
fig, ax = plt.subplots(figsize=(6.0, 3.4))
bars = ax.bar(sub['Tahap'], sub['FAR (%)'], color='#4C72B0')
ax.set_ylabel('FAR (%)'); ax.set_title('FAR trafik nyata (benign) per durasi observasi')
for b, v in zip(bars, sub['FAR (%)']):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.4f}%', ha='center', fontweight='bold')
ax.set_ylim(0, max(0.6, sub['FAR (%)'].max()*1.4))
plt.tight_layout(); plt.show()
print('FAR rendah & konsisten -> detektor tidak cerewet pada trafik normal nyata.')

## 9. Diagram Alur Pipeline (Paket -> Keputusan)

In [ ]:
# Diagram alur sederhana pakai matplotlib (tanpa dependensi tambahan)
fig, ax = plt.subplots(figsize=(11, 2.2))
ax.axis('off')
steps = ['Paket jaringan\n(ens5)', 'tcpdump\n-> .pcap', 'NFStream\nflow + statistik',
         '9 fitur SFM\n(+konversi satuan)', 'z-score\n(scaler latih)', 'XGBoost\npredict',
         'Label:\nnormal / attack']
n = len(steps); x = np.linspace(0.02, 0.98, n)
for i, (xi, s) in enumerate(zip(x, steps)):
    ax.add_patch(plt.Rectangle((xi-0.06, 0.35), 0.12, 0.3, fc='#EAF0F7', ec='#4C72B0', lw=1.5))
    ax.text(xi, 0.5, s, ha='center', va='center', fontsize=9)
    if i < n-1:
        ax.annotate('', xy=(x[i+1]-0.065, 0.5), xytext=(xi+0.065, 0.5),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=1.4))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Alur validasi trafik nyata: dari paket ke keputusan', fontsize=11)
plt.tight_layout(); plt.show()

## 9b. Galeri Gambar Siap-Pakai (dari `figure-q1/`)

Gambar-gambar publikasi yang sudah dibuat sebelumnya (via `make_figures.py`). Bila folder
`figure-q1/` tersedia, sel di bawah menampilkannya langsung sehingga mudah disalin ke slide.

In [ ]:
from matplotlib import image as mpimg

FIG_CANDIDATES = [os.path.join(DATA_DIR, 'figure-q1'), 'figure-q1', '../figure-q1']
FIG_DIR = next((d for d in FIG_CANDIDATES if os.path.isdir(d)), None)
print('FIG_DIR =', os.path.abspath(FIG_DIR) if FIG_DIR else 'tidak ditemukan')

figs = [
    ('fig1_generalization_gap.png', 'Celah generalisasi in-domain vs lintas-jaringan'),
    ('fig2_fewshot_curve.png',      'Kurva few-shot: 1% label target memulihkan MCC'),
    ('fig3_alignment.png',          'Strategi penyelarasan (baseline / CORAL / joint)'),
    ('fig4_functional_evasion.png', 'Functional-preserving evasion vs FGSM tak-terbatas'),
    ('fig5_adaptive_whitebox.png',  'Adaptive white-box: ketahanan runtuh'),
    ('fig6_deployment_blueprint.png','Blueprint penyebaran (edge / real-traffic)'),
]

if FIG_DIR:
    for fname, cap in figs:
        p = os.path.join(FIG_DIR, fname)
        if os.path.exists(p):
            img = mpimg.imread(p)
            fig, ax = plt.subplots(figsize=(7.5, 7.5*img.shape[0]/img.shape[1]))
            ax.imshow(img); ax.axis('off'); ax.set_title(cap, fontsize=10)
            plt.tight_layout(); plt.show()
        else:
            print('  (lewati, tak ada):', fname)
else:
    print('Folder figure-q1/ tidak ditemukan; lewati galeri. Diagram di sel sebelumnya tetap tersedia.')

## 10. Rangkuman Temuan Utama (untuk slide penutup)

1. **SFM valid**: 9 fitur irisan cukup ekspresif (joint training ~ in-domain di kedua jaringan).
2. **NIDS single-source runtuh lintas-jaringan** (MCC ~0), akibat *distribution shift* — bukan fitur.
3. **Kalibrasi minimal 1% label target** (atau *mixup* tanpa label) memulihkan MCC ke 0.65-0.91.
4. **Adversarial training menipu**: kuat vs transfer, runtuh vs adaptive white-box (koreksi jujur).
5. **Model ringan** (~2.9 MB, ratusan us/flow, ribuan flow/detik di 1 vCPU) -> layak *edge*.
6. **Validasi trafik nyata AWS**: FAR rendah & stabil pada trafik benign (S0 0%, D1 0.38%),
   membuktikan detektor tidak cerewet di dunia nyata.

*Semua angka berasal dari eksperimen nyata dan dilaporkan apa adanya.*